In [6]:
import sqlite3
import pandas as pd
import numpy as np
import warnings
import time
import math
from datetime import datetime
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('future.no_silent_downcasting', True)

DB_PATH = r'C:\Users\Sarthak\Documents\ML\fighter-beta\mma_fighters.db'
print("✔ Imports complete")

conn = sqlite3.connect(DB_PATH)

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(f"✔ Connected. Tables: {len(tables)}")

fights_count = pd.read_sql("SELECT COUNT(*) as c FROM fights_v2", conn)['c'].iloc[0]
print(f"✔ Total fights in DB: {fights_count}")

✔ Imports complete
✔ Connected. Tables: 10
✔ Total fights in DB: 11127


In [7]:


# ============================================================
# CELL 3: Helper Functions
# ============================================================

def parse_reach(r):
    if pd.isna(r) or r == '--': return None
    try: return float(r.replace('"', ''))
    except: return None

def parse_height(h):
    if pd.isna(h) or h == '--': return None
    try:
        parts = h.replace('"', '').split("'")
        return int(parts[0]) * 12 + int(parts[1])
    except: return None

def parse_age(dob, fight_date):
    if pd.isna(dob) or pd.isna(fight_date): return None
    try:
        birth = datetime.strptime(dob, "%b %d, %Y")
        fight = datetime.strptime(fight_date, "%Y-%m-%d")
        return (fight - birth).days / 365.25
    except: return None

def parse_fight_duration(ending_round, ending_time):
    try:
        mins, secs = ending_time.split(':')
        final_round_minutes = int(mins) + int(secs) / 60
        return ((ending_round - 1) * 5) + final_round_minutes
    except: return 15.0

# Time decay: 1.5 year half-life
LAM = math.log(2) / (1.5 * 365)

def time_decay_weights(dates, as_of_date, lam=LAM):
    as_of = datetime.strptime(as_of_date, "%Y-%m-%d")
    weights = []
    for d in dates:
        fight_dt = datetime.strptime(d, "%Y-%m-%d")
        days_ago = (as_of - fight_dt).days
        w = np.exp(-lam * max(days_ago, 0))
        weights.append(w)
    weights = np.array(weights)
    return weights / weights.sum() if weights.sum() > 0 else weights

def kish_effective_n(weights):
    if weights.sum() == 0: return 0
    return (weights.sum() ** 2) / (weights ** 2).sum()

def normalize_weight_class(wc):
    if pd.isna(wc): return None
    wc = wc.strip()
    mens_classes = [
        'Heavyweight', 'Light Heavyweight', 'Middleweight',
        'Welterweight', 'Lightweight', 'Featherweight',
        'Bantamweight', 'Flyweight', 'Catch Weight'
    ]
    for base in mens_classes:
        if base in wc and "Women's" not in wc:
            return f'{base} Bout'
    womens_classes = [
        "Women's Atomweight", "Women's Bantamweight",
        "Women's Featherweight", "Women's Flyweight",
        "Women's Strawweight"
    ]
    for base in womens_classes:
        if wc == f"{base} Bout":
            return f"{base} Bout"
    return None

print(f"✔ Lambda (1.5yr half-life): {LAM:.6f}")
print("✔ Helper functions ready")



✔ Lambda (1.5yr half-life): 0.001266
✔ Helper functions ready


In [8]:

# ============================================================
# CELL 4: Pre-Cache Fight Data (Poisson-Gamma & Beta-Binomial smoothing)
# ============================================================

print("Fetching all fight stats (no date filter)...")
start = time.time()

all_fight_stats_raw = pd.read_sql("""
    SELECT
        f.fight_id,
        f.event_date,
        f.weight_class,
        f.ending_round,
        f.ending_time,
        f.method,
        ff.fighter_id,
        SUM(frs.sig_str_landed)    as sig_str_landed,
        SUM(frs.sig_str_attempted) as sig_str_attempted,
        SUM(frs.td_landed)         as td_landed,
        SUM(frs.td_attempted)      as td_attempted,
        SUM(frs.sub_attempts)      as sub_attempts,
        SUM(frs.ctrl_seconds)      as ctrl_seconds,
        SUM(frs.knockdowns)        as knockdowns
    FROM fight_round_stats_v2 frs
    JOIN fight_rounds_v2 fr      ON SUBSTR(frs.round_id, 1, INSTR(frs.round_id, ':') - 1) = fr.fight_id
    JOIN fights_v2 f             ON fr.fight_id   = f.fight_id
    JOIN fight_fighters_v2 ff    ON f.fight_id    = ff.fight_id
                                AND frs.fighter_id = ff.fighter_id
    GROUP BY f.fight_id, ff.fighter_id
""", conn)

all_fight_stats_raw['actual_minutes'] = all_fight_stats_raw.apply(
    lambda r: parse_fight_duration(r['ending_round'], r['ending_time']), axis=1
)
all_fight_stats_raw = all_fight_stats_raw.replace([np.inf, -np.inf], np.nan).fillna(0)

# ── Weight-class priors ────────────────────────────────────
print("Computing weight-class priors...")

all_fight_stats_raw['wc_norm'] = all_fight_stats_raw['weight_class'].apply(normalize_weight_class)
valid = all_fight_stats_raw[all_fight_stats_raw['wc_norm'].notna()].copy()

pg_stats = {
    'sig_str_landed': 0.7,
    'td_landed':      7.0,
    'sub_attempts':   12.0,
    'knockdowns':     20.0,
    'ctrl_seconds':   2.0,
}

pg_tau_overrides = {
    'Heavyweight Bout':              {'td_landed': 5.0, 'ctrl_seconds': 1.5},
    'Light Heavyweight Bout':        {'ctrl_seconds': 1.5},
    "Women's Bantamweight Bout":     {'ctrl_seconds': 1.5},
    "Women's Featherweight Bout":    {'ctrl_seconds': 1.5},
}

def get_pg_tau(wc_norm, stat):
    if wc_norm and wc_norm in pg_tau_overrides:
        if stat in pg_tau_overrides[wc_norm]:
            return pg_tau_overrides[wc_norm][stat]
    return pg_stats.get(stat, 1.0)

wc_rates     = {}
global_rates = {}

for stat in pg_stats:
    global_total       = valid[stat].sum()
    global_mins        = valid['actual_minutes'].sum()
    global_rates[stat] = global_total / global_mins if global_mins > 0 else 0
    for wc, grp in valid.groupby('wc_norm'):
        if wc not in wc_rates:
            wc_rates[wc] = {}
        total = grp[stat].sum()
        mins  = grp['actual_minutes'].sum()
        wc_rates[wc][stat] = total / mins if mins > 0 else global_rates[stat]

bb_stats = {
    'str_acc':  {'num': 'sig_str_landed', 'den': 'sig_str_attempted', 'tau': 0.7},
    'td_acc':   {'num': 'td_landed',      'den': 'td_attempted',      'tau': 7.0},
    'sub_land': {'num': 'sub_attempts',   'den': 'sub_attempts',      'tau': 9.0},
    'ctrl':     {'num': 'ctrl_seconds',   'den': 'actual_minutes_sec','tau': 2.0},
}

bb_tau_overrides = {
    'Featherweight Bout':            {'sub_land': 3.0},
    "Women's Featherweight Bout":    {'sub_land': 3.0, 'ctrl': 1.5},
    'Heavyweight Bout':              {'ctrl': 1.5},
    'Light Heavyweight Bout':        {'ctrl': 1.5},
    "Women's Bantamweight Bout":     {'ctrl': 1.5},
}

def get_bb_tau(wc_norm, stat):
    if wc_norm and wc_norm in bb_tau_overrides:
        if stat in bb_tau_overrides[wc_norm]:
            return bb_tau_overrides[wc_norm][stat]
    return bb_stats[stat]['tau']

wc_bb_priors     = {}
global_bb_priors = {}

all_fight_stats_raw['actual_minutes_sec'] = all_fight_stats_raw['actual_minutes'] * 60
valid = all_fight_stats_raw[all_fight_stats_raw['wc_norm'].notna()].copy()

for stat, cfg in bb_stats.items():
    if cfg['den'] == 'actual_minutes_sec':
        global_num = valid[cfg['num']].sum()
        global_den = (valid['actual_minutes'] * 60).sum()
    else:
        global_num = valid[cfg['num']].sum()
        global_den = valid[cfg['den']].sum()
    global_bb_priors[stat] = global_num / global_den if global_den > 0 else 0.5
    for wc, grp in valid.groupby('wc_norm'):
        if wc not in wc_bb_priors:
            wc_bb_priors[wc] = {}
        if cfg['den'] == 'actual_minutes_sec':
            num = grp[cfg['num']].sum()
            den = (grp['actual_minutes'] * 60).sum()
        else:
            num = grp[cfg['num']].sum()
            den = grp[cfg['den']].sum()
        wc_bb_priors[wc][stat] = num / den if den > 0 else global_bb_priors[stat]

print(f"✓ Weight class priors computed for {len(wc_rates)} classes")

# ── Apply smoothing ────────────────────────────────────────
print("Applying Beta-Binomial smoothing first, then Poisson-Gamma...")

all_fight_stats = all_fight_stats_raw.copy()

def get_wc_rate(wc, stat):
    wc_norm = normalize_weight_class(wc) if wc else None
    if wc_norm and wc_norm in wc_rates and stat in wc_rates[wc_norm]:
        return wc_rates[wc_norm][stat]
    return global_rates.get(stat, 0)

def get_wc_bb(wc, stat):
    wc_norm = normalize_weight_class(wc) if wc else None
    if wc_norm and wc_norm in wc_bb_priors and stat in wc_bb_priors[wc_norm]:
        return wc_bb_priors[wc_norm][stat]
    return global_bb_priors.get(stat, 0.5)

# Beta-Binomial first
for stat, cfg in bb_stats.items():
    smoothed = []
    for _, row in all_fight_stats.iterrows():
        wc_norm = normalize_weight_class(row['weight_class'])
        wc_rate = get_wc_bb(row['weight_class'], stat)
        tau     = get_bb_tau(wc_norm, stat)
        x       = row[cfg['num']]
        n       = row['actual_minutes'] * 60 if cfg['den'] == 'actual_minutes_sec' else row[cfg['den']]
        smoothed.append(wc_rate if n == 0 else (wc_rate * tau + x) / (tau + n))
    all_fight_stats[f'{stat}_smooth'] = smoothed

# Poisson-Gamma second
for stat in pg_stats:
    smoothed = []
    for _, row in all_fight_stats.iterrows():
        wc_norm  = normalize_weight_class(row['weight_class'])
        t        = max(row['actual_minutes'], 0.1)
        wc_rate  = get_wc_rate(row['weight_class'], stat)
        tau      = get_pg_tau(wc_norm, stat)
        x        = row[stat]
        post_rate = (wc_rate * tau + x) / (tau + t)
        smoothed.append(t * post_rate)
    all_fight_stats[f'{stat}_smooth'] = smoothed

# Per-minute rates
t = all_fight_stats['actual_minutes'].clip(lower=0.1)
all_fight_stats['slpm']              = all_fight_stats['sig_str_landed_smooth'] / t
all_fight_stats['str_acc']           = all_fight_stats['str_acc_smooth']
all_fight_stats['td_acc']            = all_fight_stats['td_acc_smooth']
all_fight_stats['td_avg']            = all_fight_stats['td_landed_smooth']    / (t / 15)
all_fight_stats['sub_avg']           = all_fight_stats['sub_attempts_smooth'] / (t / 15)
all_fight_stats['ctrl_time_per_min'] = all_fight_stats['ctrl_smooth'] / t
all_fight_stats['kd_per_min']        = all_fight_stats['knockdowns_smooth']   / t

all_fight_stats = all_fight_stats.replace([np.inf, -np.inf], np.nan).fillna(0)
all_fight_stats = all_fight_stats.sort_values('event_date').reset_index(drop=True)

# Rebuild dicts
fighter_fights_dict = {
    fid: grp.sort_values('event_date').reset_index(drop=True)
    for fid, grp in all_fight_stats.groupby('fighter_id')
}

all_opponents = pd.read_sql("""
    SELECT ff1.fight_id, ff1.fighter_id, ff2.fighter_id as opponent_id
    FROM fight_fighters_v2 ff1
    JOIN fight_fighters_v2 ff2
        ON ff1.fight_id = ff2.fight_id
       AND ff1.fighter_id != ff2.fighter_id
""", conn)

opponents_dict = {
    (row['fight_id'], row['fighter_id']): row['opponent_id']
    for _, row in all_opponents.iterrows()
}

print(f"Done in {time.time()-start:.1f}s")
print(f"✓ Fighters cached: {len(fighter_fights_dict)}")
print(f"✓ Fight records:   {len(all_fight_stats)}")
print(f"✓ Opponent lookups:{len(opponents_dict)}")


Fetching all fight stats (no date filter)...
Computing weight-class priors...
✓ Weight class priors computed for 13 classes
Applying Beta-Binomial smoothing first, then Poisson-Gamma...
Done in 14.5s
✓ Fighters cached: 3760
✓ Fight records:   21568
✓ Opponent lookups:22254


In [9]:

# ============================================================
# CELL 4b: Pre-Cache Strike Breakdown
# ============================================================

print("Fetching strike breakdown data...")
start = time.time()

strike_breakdown = pd.read_sql("""
    SELECT
        f.fight_id,
        f.event_date,
        f.weight_class,
        f.ending_round,
        f.ending_time,
        ff.fighter_id,
        SUM(ss.head_landed)         as head_landed,
        SUM(ss.head_attempted)      as head_attempted,
        SUM(ss.body_landed)         as body_landed,
        SUM(ss.body_attempted)      as body_attempted,
        SUM(ss.leg_landed)          as leg_landed,
        SUM(ss.leg_attempted)       as leg_attempted,
        SUM(ss.distance_landed)     as distance_landed,
        SUM(ss.distance_attempted)  as distance_attempted,
        SUM(ss.clinch_landed)       as clinch_landed,
        SUM(ss.clinch_attempted)    as clinch_attempted,
        SUM(ss.ground_landed)       as ground_landed,
        SUM(ss.ground_attempted)    as ground_attempted
    FROM fight_round_sig_strikes_v2 ss
    JOIN fight_rounds_v2 fr      ON ss.round_id   = fr.round_id
    JOIN fights_v2 f             ON fr.fight_id   = f.fight_id
    JOIN fight_fighters_v2 ff    ON f.fight_id    = ff.fight_id
                                AND ss.fighter_id  = ff.fighter_id
    GROUP BY f.fight_id, ff.fighter_id
""", conn)

strike_breakdown['actual_minutes'] = strike_breakdown.apply(
    lambda r: parse_fight_duration(r['ending_round'], r['ending_time']), axis=1
)

for pos in ['head', 'body', 'leg', 'distance', 'clinch', 'ground']:
    strike_breakdown[f'{pos}_lpm'] = (
        strike_breakdown[f'{pos}_landed'] / strike_breakdown['actual_minutes']
    )
    strike_breakdown[f'{pos}_acc'] = (
        strike_breakdown[f'{pos}_landed'] / strike_breakdown[f'{pos}_attempted']
    )

strike_breakdown['total_sig_landed'] = (
    strike_breakdown['head_landed'] +
    strike_breakdown['body_landed'] +
    strike_breakdown['leg_landed']
)

strike_breakdown = strike_breakdown.replace([np.inf, -np.inf], np.nan).fillna(0)
strike_breakdown = strike_breakdown.sort_values('event_date').reset_index(drop=True)

strike_breakdown_dict = {
    fid: grp.sort_values('event_date').reset_index(drop=True)
    for fid, grp in strike_breakdown.groupby('fighter_id')
}

print(f"Done in {time.time()-start:.1f}s")
print(f"✓ Strike breakdown records: {len(strike_breakdown)}")
print(f"✓ Fighters with strike data: {len(strike_breakdown_dict)}")


Fetching strike breakdown data...
Done in 2.1s
✓ Strike breakdown records: 21568
✓ Fighters with strike data: 3760


In [10]:

# ============================================================
# CELL 4c: Pre-Cache Strike Defense
# ============================================================

print("Building strike defense lookup...")
start = time.time()

opp_strike_rows = []
for _, row in strike_breakdown.iterrows():
    opp_id = opponents_dict.get((row['fight_id'], row['fighter_id']))
    if opp_id:
        opp_strike_rows.append({
            'fight_id':         row['fight_id'],
            'event_date':       row['event_date'],
            'fighter_id':       opp_id,
            'head_allowed':     row['head_lpm'],
            'body_allowed':     row['body_lpm'],
            'leg_allowed':      row['leg_lpm'],
            'distance_allowed': row['distance_lpm'],
            'clinch_allowed':   row['clinch_lpm'],
            'ground_allowed':   row['ground_lpm'],
        })

strike_defense_df = pd.DataFrame(opp_strike_rows).sort_values('event_date').reset_index(drop=True)

strike_defense_dict = {
    fid: grp.sort_values('event_date').reset_index(drop=True)
    for fid, grp in strike_defense_df.groupby('fighter_id')
}

print(f"Done in {time.time()-start:.1f}s")
print(f"✔ Defense records: {len(strike_defense_df)}")
print(f"✔ Fighters with defense data: {len(strike_defense_dict)}")


Building strike defense lookup...
Done in 1.9s
✔ Defense records: 21568
✔ Fighters with defense data: 3761


In [11]:

# ============================================================
# CELL 4d: Pre-Cache Fighter Results
# ============================================================

print("Building fighter results cache...")

results_raw = pd.read_sql("""
    SELECT
        ff.fighter_id,
        ff.fight_id,
        ff.result,
        f.method
    FROM fight_fighters_v2 ff
    JOIN fights_v2 f ON ff.fight_id = f.fight_id
""", conn)

fighter_results_dict = {}
for _, row in results_raw.iterrows():
    result = row['result']
    tagged = 'ko_win' if result == 'win' and row['method'] == 'KO/TKO' else result
    fighter_results_dict[(row['fighter_id'], row['fight_id'])] = tagged

print(f"✔ Results cached: {len(fighter_results_dict)}")


Building fighter results cache...
✔ Results cached: 22254


In [12]:


# ============================================================
# CELL 4e: Pre-Cache Round 1 Stats
# ============================================================

print("Fetching Round 1 stats...")
start = time.time()

r1_stats = pd.read_sql("""
    SELECT
        f.fight_id,
        f.event_date,
        f.weight_class,
        f.ending_round,
        f.ending_time,
        f.method,
        ff.fighter_id,
        frs.sig_str_landed    as r1_sig_str_landed,
        frs.sig_str_attempted as r1_sig_str_attempted,
        frs.ctrl_seconds      as r1_ctrl_seconds,
        frs.td_landed         as r1_td_landed,
        frs.td_attempted      as r1_td_attempted,
        frs.knockdowns        as r1_knockdowns,
        frs.reversals         as r1_reversals,
        frs.sub_attempts      as r1_sub_attempts
    FROM fight_round_stats_v2 frs
    JOIN fights_v2 f          ON SUBSTR(frs.round_id, 1, INSTR(frs.round_id, ':') - 1) = f.fight_id
    JOIN fight_fighters_v2 ff ON f.fight_id    = ff.fight_id
                             AND frs.fighter_id = ff.fighter_id
    WHERE CAST(SUBSTR(frs.round_id, INSTR(frs.round_id, ':') + 1) AS INTEGER) = 1
""", conn)

r1_sig_strikes = pd.read_sql("""
    SELECT
        f.fight_id,
        ss.fighter_id,
        ss.head_landed    as r1_head_landed,
        ss.head_attempted as r1_head_attempted,
        ss.body_landed    as r1_body_landed,
        ss.body_attempted as r1_body_attempted,
        ss.leg_landed     as r1_leg_landed,
        ss.leg_attempted  as r1_leg_attempted,
        ss.clinch_landed  as r1_clinch_landed,
        ss.clinch_attempted as r1_clinch_attempted,
        ss.distance_landed  as r1_distance_landed,
        ss.ground_landed    as r1_ground_landed
    FROM fight_round_sig_strikes_v2 ss
    JOIN fight_rounds_v2 fr ON ss.round_id = fr.round_id
    JOIN fights_v2 f        ON fr.fight_id = f.fight_id
    WHERE CAST(SUBSTR(ss.round_id, INSTR(ss.round_id, ':') + 1) AS INTEGER) = 1
""", conn)

r1_stats = r1_stats.merge(r1_sig_strikes, on=['fight_id', 'fighter_id'], how='left')
r1_sig_cols = [c for c in r1_sig_strikes.columns if c.startswith('r1_')]
r1_stats[r1_sig_cols] = r1_stats[r1_sig_cols].fillna(0)

r1_stats['r1_minutes'] = r1_stats.apply(
    lambda r: min(parse_fight_duration(r['ending_round'], r['ending_time']), 5.0), axis=1
)
r1_stats['wc_norm'] = r1_stats['weight_class'].apply(normalize_weight_class)
r1_stats = r1_stats.replace([np.inf, -np.inf], np.nan).fillna(0)

r1_valid = r1_stats[r1_stats['wc_norm'].notna()].copy()

# R1 Beta-Binomial
r1_bb_stats = {
    'r1_td_acc':   {'num': 'r1_td_landed',   'den': 'r1_td_attempted', 'tau': 9.0},
    'r1_ctrl':     {'num': 'r1_ctrl_seconds', 'den': 'r1_minutes_sec',  'tau': 1.0},
    'r1_sub_land': {'num': 'r1_sub_attempts', 'den': 'r1_sub_attempts', 'tau': 7.0},
}

r1_bb_tau_overrides = {
    'Featherweight Bout':         {'r1_sub_land': 3.0},
    "Women's Featherweight Bout": {'r1_sub_land': 3.0},
    'Heavyweight Bout':           {'r1_ctrl': 1.5},
    'Light Heavyweight Bout':     {'r1_ctrl': 1.5},
}

def get_r1_bb_tau(wc_norm, stat):
    if wc_norm and wc_norm in r1_bb_tau_overrides:
        if stat in r1_bb_tau_overrides[wc_norm]:
            return r1_bb_tau_overrides[wc_norm][stat]
    return r1_bb_stats[stat]['tau']

r1_stats['r1_minutes_sec'] = r1_stats['r1_minutes'] * 60
r1_valid = r1_stats[r1_stats['wc_norm'].notna()].copy()

r1_wc_bb     = {}
r1_global_bb = {}

for stat, cfg in r1_bb_stats.items():
    if cfg['den'] == 'r1_minutes_sec':
        global_num = r1_valid[cfg['num']].sum()
        global_den = (r1_valid['r1_minutes'] * 60).sum()
    else:
        global_num = r1_valid[cfg['num']].sum()
        global_den = r1_valid[cfg['den']].sum()
    r1_global_bb[stat] = global_num / global_den if global_den > 0 else 0.5
    for wc, grp in r1_valid.groupby('wc_norm'):
        if wc not in r1_wc_bb:
            r1_wc_bb[wc] = {}
        if cfg['den'] == 'r1_minutes_sec':
            num = grp[cfg['num']].sum()
            den = (grp['r1_minutes'] * 60).sum()
        else:
            num = grp[cfg['num']].sum()
            den = grp[cfg['den']].sum()
        r1_wc_bb[wc][stat] = num / den if den > 0 else r1_global_bb[stat]

# R1 Poisson-Gamma
r1_pg_stats = {
    'r1_sig_str_landed': 0.7,
    'r1_td_landed':      9.0,
    'r1_sub_attempts':   15.0,
    'r1_knockdowns':     12.0,
    'r1_reversals':      42.0,
    'r1_head_landed':    0.7,
    'r1_body_landed':    2.5,
    'r1_leg_landed':     1.7,
    'r1_clinch_landed':  5.0,
}

r1_pg_tau_overrides = {
    'Flyweight Bout':         {'r1_reversals': 22.0},
    "Women's Flyweight Bout": {'r1_reversals': 22.0},
    'Light Heavyweight Bout': {'r1_head_landed': 0.5},
    'Heavyweight Bout':       {'r1_td_landed': 4.0},
}

def get_r1_pg_tau(wc_norm, stat):
    if wc_norm and wc_norm in r1_pg_tau_overrides:
        if stat in r1_pg_tau_overrides[wc_norm]:
            return r1_pg_tau_overrides[wc_norm][stat]
    return r1_pg_stats.get(stat, 1.0)

r1_wc_rates     = {}
r1_global_rates = {}

for stat in r1_pg_stats:
    total = r1_valid[stat].sum()
    mins  = r1_valid['r1_minutes'].sum()
    r1_global_rates[stat] = total / mins if mins > 0 else 0
    for wc, grp in r1_valid.groupby('wc_norm'):
        if wc not in r1_wc_rates:
            r1_wc_rates[wc] = {}
        wc_total = grp[stat].sum()
        wc_mins  = grp['r1_minutes'].sum()
        r1_wc_rates[wc][stat] = wc_total / wc_mins if wc_mins > 0 else r1_global_rates[stat]

def get_r1_wc_rate(wc, stat):
    wc_n = normalize_weight_class(wc) if wc else None
    if wc_n and wc_n in r1_wc_rates and stat in r1_wc_rates[wc_n]:
        return r1_wc_rates[wc_n][stat]
    return r1_global_rates.get(stat, 0)

def get_r1_wc_bb(wc, stat):
    wc_n = normalize_weight_class(wc) if wc else None
    if wc_n and wc_n in r1_wc_bb and stat in r1_wc_bb[wc_n]:
        return r1_wc_bb[wc_n][stat]
    return r1_global_bb.get(stat, 0.5)

# Apply BB smoothing first
for stat, cfg in r1_bb_stats.items():
    smoothed = []
    for _, row in r1_stats.iterrows():
        wc_norm = normalize_weight_class(row['weight_class'])
        wc_rate = get_r1_wc_bb(row['weight_class'], stat)
        tau     = get_r1_bb_tau(wc_norm, stat)
        x       = row[cfg['num']]
        n       = row['r1_minutes'] * 60 if cfg['den'] == 'r1_minutes_sec' else row[cfg['den']]
        smoothed.append(wc_rate if n == 0 else (wc_rate * tau + x) / (tau + n))
    r1_stats[f'{stat}_smooth'] = smoothed

# Apply PG smoothing second
for stat in r1_pg_stats:
    smoothed = []
    for _, row in r1_stats.iterrows():
        wc_norm = normalize_weight_class(row['weight_class'])
        t       = max(row['r1_minutes'], 0.1)
        wc_rate = get_r1_wc_rate(row['weight_class'], stat)
        tau     = get_r1_pg_tau(wc_norm, stat)
        x       = row[stat]
        post    = (wc_rate * tau + x) / (tau + t)
        smoothed.append(t * post)
    r1_stats[f'{stat}_smooth'] = smoothed

t = r1_stats['r1_minutes'].clip(lower=0.1)
r1_stats['r1_slpm']           = r1_stats['r1_sig_str_landed_smooth'] / t
r1_stats['r1_td_per_min']     = r1_stats['r1_td_landed_smooth'] / t
r1_stats['r1_td_acc']         = r1_stats['r1_td_acc_smooth']
r1_stats['r1_ctrl_per_min']   = r1_stats['r1_ctrl_smooth']
r1_stats['r1_kd_per_min']     = r1_stats['r1_knockdowns_smooth'] / t
r1_stats['r1_rev_per_min']    = r1_stats['r1_reversals_smooth'] / t
r1_stats['r1_sub_per_min']    = r1_stats['r1_sub_attempts_smooth'] / t
r1_stats['r1_td_att_per_min'] = r1_stats['r1_td_attempted'] / t
r1_stats['r1_head_lpm']       = r1_stats['r1_head_landed_smooth'] / t
r1_stats['r1_body_lpm']       = r1_stats['r1_body_landed_smooth'] / t
r1_stats['r1_leg_lpm']        = r1_stats['r1_leg_landed_smooth'] / t
r1_stats['r1_clinch_lpm']     = r1_stats['r1_clinch_landed_smooth'] / t

r1_stats = r1_stats.replace([np.inf, -np.inf], np.nan).fillna(0)
r1_stats = r1_stats.sort_values('event_date').reset_index(drop=True)

r1_stats_dict = {
    fid: grp.sort_values('event_date').reset_index(drop=True)
    for fid, grp in r1_stats.groupby('fighter_id')
}

print(f"Done in {time.time()-start:.1f}s")
print(f"✔ R1 records: {len(r1_stats)}")
print(f"✔ Fighters with R1 data: {len(r1_stats_dict)}")



Fetching Round 1 stats...
Done in 16.4s
✔ R1 records: 21568
✔ Fighters with R1 data: 3760


In [13]:
# ============================================================
# CELL 4f: Weight Class Priors
# ============================================================

print("Building weight class priors...")

STATS_FOR_PRIORS = ['slpm', 'str_acc', 'td_acc', 'td_avg', 'sub_avg',
                    'ctrl_time_per_min', 'kd_per_min']

all_fight_stats['weight_class_norm'] = all_fight_stats['weight_class'].apply(normalize_weight_class)
valid_stats = all_fight_stats[all_fight_stats['weight_class_norm'].notna()].copy()

wc_priors     = {}
global_priors = {}

for stat in STATS_FOR_PRIORS:
    global_median = float(np.median(valid_stats[stat].values))
    global_mad    = float(np.median(np.abs(valid_stats[stat].values - global_median)))
    global_priors[stat] = {'mean': global_median, 'mad': max(global_mad, 0.01)}
    for wc, grp in valid_stats.groupby('weight_class_norm'):
        if wc not in wc_priors:
            wc_priors[wc] = {}
        vals   = grp[stat].values
        median = float(np.median(vals))
        mad    = float(np.median(np.abs(vals - median)))
        wc_priors[wc][stat] = {'mean': median, 'mad': max(mad, 0.01)}

print(f"✔ Weight classes with priors: {len(wc_priors)}")
print(f"✔ Global fallback priors: {len(global_priors)} stats")

Building weight class priors...
✔ Weight classes with priors: 13
✔ Global fallback priors: 7 stats


In [14]:

# ============================================================
# CELL 4g: Pre-Cache Per-Fight AdjPerf Z-Scores
# ============================================================

WINDOW = 5
K_MEAN = 4.0
K_MAD  = 4.0

def bayesian_smooth(observed, n_eff, population_mean, k):
    w = n_eff / (n_eff + k)
    return w * observed + (1 - w) * population_mean

print("Pre-caching per-fight AdjPerf z-scores (core + strike + R1)...")
start = time.time()

CORE_STATS           = ['slpm', 'str_acc', 'td_acc', 'td_avg', 'sub_avg',
                         'ctrl_time_per_min', 'kd_per_min']
STRIKE_STATS_FOR_CACHE = ['head_lpm', 'body_acc', 'distance_acc',
                           'distance_lpm', 'ground_allowed',
                           'head_acc', 'distance_allowed']
R1_STATS_FOR_CACHE   = ['r1_slpm', 'r1_rev_per_min']

all_fights_sorted = all_fight_stats.sort_values('event_date').reset_index(drop=True)

pop_means_core = {s: float(np.median(all_fight_stats[s].values)) for s in CORE_STATS}
pop_mads_core  = {s: float(np.median(np.abs(
    all_fight_stats[s].values - pop_means_core[s]
))) for s in CORE_STATS}

pop_means_strike = {}
pop_mads_strike  = {}
for s in STRIKE_STATS_FOR_CACHE:
    src = strike_breakdown if s in strike_breakdown.columns else strike_defense_df
    pop_means_strike[s] = float(src[s].median()) if s in src.columns else 0.0
    pop_mads_strike[s]  = float(max(np.median(np.abs(
        src[s].values - src[s].median()
    )), 0.01)) if s in src.columns else 1.0

pop_means_r1 = {}
pop_mads_r1  = {}
for s in R1_STATS_FOR_CACHE:
    pop_means_r1[s] = float(r1_stats[s].median()) if s in r1_stats.columns else 0.0
    pop_mads_r1[s]  = float(max(np.median(np.abs(
        r1_stats[s].values - r1_stats[s].median()
    )), 0.01)) if s in r1_stats.columns else 1.0

adjperf_rows = []

for idx, fight_row in all_fights_sorted.iterrows():
    if idx % 1000 == 0:
        print(f"  {idx}/{len(all_fights_sorted)}...")

    fid       = fight_row['fighter_id']
    fdate     = fight_row['event_date']
    fid_fight = fight_row['fight_id']
    opp_id    = opponents_dict.get((fid_fight, fid))
    if not opp_id:
        continue

    fighter_hist = fighter_fights_dict.get(fid)
    opp_hist     = fighter_fights_dict.get(opp_id)
    if fighter_hist is None or opp_hist is None:
        continue

    fighter_prev = fighter_hist[fighter_hist['event_date'] < fdate].tail(WINDOW)
    opp_prev     = opp_hist[opp_hist['event_date'] < fdate]
    if len(fighter_prev) == 0:
        continue

    row = {'fighter_id': fid, 'fight_id': fid_fight, 'event_date': fdate}
    f_weights = time_decay_weights(fighter_prev['event_date'].tolist(), fdate)
    f_n_eff   = kish_effective_n(f_weights)

    for stat in CORE_STATS:
        dec_avg  = np.average(fighter_prev[stat].values, weights=f_weights)
        smoothed = bayesian_smooth(dec_avg, f_n_eff, pop_means_core[stat], K_MEAN)
        opp_allowed, opp_dates = [], []
        for _, opp_fight in opp_prev.iterrows():
            opp_opp_id = opponents_dict.get((opp_fight['fight_id'], opp_id))
            if opp_opp_id and opp_opp_id in fighter_fights_dict:
                opp_opp_this = fighter_fights_dict[opp_opp_id][
                    fighter_fights_dict[opp_opp_id]['fight_id'] == opp_fight['fight_id']
                ]
                if len(opp_opp_this) > 0:
                    opp_allowed.append(opp_opp_this[stat].iloc[0])
                    opp_dates.append(opp_fight['event_date'])
        if len(opp_allowed) >= 2:
            opp_w     = time_decay_weights(opp_dates, fdate)
            opp_n_eff = kish_effective_n(opp_w)
            opp_mu    = bayesian_smooth(np.average(opp_allowed, weights=opp_w),
                                        opp_n_eff, pop_means_core[stat], K_MEAN)
            opp_sigma = max(bayesian_smooth(
                float(np.median(np.abs(np.array(opp_allowed) - np.median(opp_allowed)))),
                opp_n_eff, pop_mads_core[stat], K_MAD), 0.01)
            row[f'{stat}_adjperf_snapshot'] = float(np.clip((smoothed - opp_mu) / opp_sigma, -7, 7))
        else:
            row[f'{stat}_adjperf_snapshot'] = 0.0

    for stat in STRIKE_STATS_FOR_CACHE:
        src_dict = strike_defense_dict if stat in ['ground_allowed', 'distance_allowed'] else strike_breakdown_dict
        if fid not in src_dict:
            row[f'{stat}_adjperf_snapshot'] = 0.0
            continue
        f_hist = src_dict[fid]
        f_prev = f_hist[f_hist['event_date'] < fdate].tail(WINDOW)
        if len(f_prev) == 0 or stat not in f_prev.columns:
            row[f'{stat}_adjperf_snapshot'] = 0.0
            continue
        sw       = time_decay_weights(f_prev['event_date'].tolist(), fdate)
        s_n_eff  = kish_effective_n(sw)
        smoothed = bayesian_smooth(np.average(f_prev[stat].values, weights=sw),
                                   s_n_eff, pop_means_strike[stat], K_MEAN)
        if opp_id not in src_dict:
            row[f'{stat}_adjperf_snapshot'] = 0.0
            continue
        opp_s_hist = src_dict[opp_id]
        opp_s_prev = opp_s_hist[opp_s_hist['event_date'] < fdate]
        opp_allowed, opp_dates = [], []
        for _, opp_fight in opp_s_prev.iterrows():
            opp_opp_id = opponents_dict.get((opp_fight['fight_id'], opp_id))
            if opp_opp_id and opp_opp_id in src_dict:
                opp_opp_this = src_dict[opp_opp_id][
                    src_dict[opp_opp_id]['fight_id'] == opp_fight['fight_id']
                ]
                if len(opp_opp_this) > 0 and stat in opp_opp_this.columns:
                    opp_allowed.append(opp_opp_this[stat].iloc[0])
                    opp_dates.append(opp_fight['event_date'])
        if len(opp_allowed) >= 2:
            opp_w     = time_decay_weights(opp_dates, fdate)
            opp_n_eff = kish_effective_n(opp_w)
            opp_mu    = bayesian_smooth(np.average(opp_allowed, weights=opp_w),
                                        opp_n_eff, pop_means_strike[stat], K_MEAN)
            opp_sigma = max(bayesian_smooth(
                float(np.median(np.abs(np.array(opp_allowed) - np.median(opp_allowed)))),
                opp_n_eff, pop_mads_strike[stat], K_MAD), 0.01)
            row[f'{stat}_adjperf_snapshot'] = float(np.clip((smoothed - opp_mu) / opp_sigma, -7, 7))
        else:
            row[f'{stat}_adjperf_snapshot'] = 0.0

    for stat in R1_STATS_FOR_CACHE:
        if fid not in r1_stats_dict:
            row[f'{stat}_adjperf_snapshot'] = 0.0
            continue
        f_r1 = r1_stats_dict[fid]
        f_r1_prev = f_r1[f_r1['event_date'] < fdate].tail(WINDOW)
        if len(f_r1_prev) == 0 or stat not in f_r1_prev.columns:
            row[f'{stat}_adjperf_snapshot'] = 0.0
            continue
        rw       = time_decay_weights(f_r1_prev['event_date'].tolist(), fdate)
        r_n_eff  = kish_effective_n(rw)
        smoothed = bayesian_smooth(np.average(f_r1_prev[stat].values, weights=rw),
                                   r_n_eff, pop_means_r1[stat], K_MEAN)
        if opp_id not in r1_stats_dict:
            row[f'{stat}_adjperf_snapshot'] = 0.0
            continue
        opp_r1 = r1_stats_dict[opp_id]
        opp_r1_prev = opp_r1[opp_r1['event_date'] < fdate]
        opp_allowed, opp_dates = [], []
        for _, opp_fight in opp_r1_prev.iterrows():
            opp_opp_id = opponents_dict.get((opp_fight['fight_id'], opp_id))
            if opp_opp_id and opp_opp_id in r1_stats_dict:
                opp_opp_this = r1_stats_dict[opp_opp_id][
                    r1_stats_dict[opp_opp_id]['fight_id'] == opp_fight['fight_id']
                ]
                if len(opp_opp_this) > 0 and stat in opp_opp_this.columns:
                    opp_allowed.append(opp_opp_this[stat].iloc[0])
                    opp_dates.append(opp_fight['event_date'])
        if len(opp_allowed) >= 2:
            opp_w     = time_decay_weights(opp_dates, fdate)
            opp_n_eff = kish_effective_n(opp_w)
            opp_mu    = bayesian_smooth(np.average(opp_allowed, weights=opp_w),
                                        opp_n_eff, pop_means_r1[stat], K_MEAN)
            opp_sigma = max(bayesian_smooth(
                float(np.median(np.abs(np.array(opp_allowed) - np.median(opp_allowed)))),
                opp_n_eff, pop_mads_r1[stat], K_MAD), 0.01)
            row[f'{stat}_adjperf_snapshot'] = float(np.clip((smoothed - opp_mu) / opp_sigma, -7, 7))
        else:
            row[f'{stat}_adjperf_snapshot'] = 0.0

    adjperf_rows.append(row)

adjperf_history_df = pd.DataFrame(adjperf_rows).sort_values('event_date').reset_index(drop=True)
fighter_adjperf_history = {
    fid: grp.sort_values('event_date').reset_index(drop=True)
    for fid, grp in adjperf_history_df.groupby('fighter_id')
}

print(f"Done in {time.time()-start:.1f}s")
print(f"✓ AdjPerf history cached for {len(fighter_adjperf_history)} fighters")
print(f"✓ Total records: {len(adjperf_history_df)}")



Pre-caching per-fight AdjPerf z-scores (core + strike + R1)...
  0/21568...
  1000/21568...
  2000/21568...
  3000/21568...
  4000/21568...
  5000/21568...
  6000/21568...
  7000/21568...
  8000/21568...
  9000/21568...
  10000/21568...
  11000/21568...
  12000/21568...
  13000/21568...
  14000/21568...
  15000/21568...
  16000/21568...
  17000/21568...
  18000/21568...
  19000/21568...
  20000/21568...
  21000/21568...
Done in 810.3s
✓ AdjPerf history cached for 2661 fighters
✓ Total records: 17755


In [15]:

# ============================================================
# CELL 5: Generate Base Fights
# ============================================================

EXCLUDED_METHODS = ['S-DEC', 'M-DEC', 'Overturned', 'CNC', 'DQ', 'Other', 'Decision']
MIN_PREV_FIGHTS  = 3

def generate_base_fights_filtered(start_date='2014-01-01', end_date='2026-12-31'):
    excl = "', '".join(EXCLUDED_METHODS)
    fights = pd.read_sql(f"""
        SELECT
            f.fight_id,
            f.event_date,
            f.event_name,
            f.weight_class,
            f.method,
            f.ending_round,
            f.ending_time,
            ff1.fighter_id  as fighter_1_id,
            fv1.name        as fighter_1_name,
            ff2.fighter_id  as fighter_2_id,
            fv2.name        as fighter_2_name,
            CASE WHEN ff1.result = 'win' THEN 1 ELSE 0 END as winner
        FROM fights_v2 f
        JOIN fight_fighters_v2 ff1 ON f.fight_id = ff1.fight_id AND ff1.corner = 'fighter_1'
        JOIN fight_fighters_v2 ff2 ON f.fight_id = ff2.fight_id AND ff2.corner = 'fighter_2'
        JOIN fighters_v2 fv1       ON ff1.fighter_id = fv1.fighter_id
        JOIN fighters_v2 fv2       ON ff2.fighter_id = fv2.fighter_id
        WHERE f.event_date >= '{start_date}'
          AND f.event_date <= '{end_date}'
          AND f.method NOT IN ('{excl}')
        ORDER BY f.event_date
    """, conn)

    valid = []
    for _, fight in fights.iterrows():
        f1_id, f2_id, fdate = fight['fighter_1_id'], fight['fighter_2_id'], fight['event_date']
        f1_prev = len(fighter_fights_dict[f1_id][fighter_fights_dict[f1_id]['event_date'] < fdate]) \
                  if f1_id in fighter_fights_dict else 0
        f2_prev = len(fighter_fights_dict[f2_id][fighter_fights_dict[f2_id]['event_date'] < fdate]) \
                  if f2_id in fighter_fights_dict else 0
        if f1_prev >= MIN_PREV_FIGHTS and f2_prev >= MIN_PREV_FIGHTS:
            valid.append(fight)

    filtered = pd.DataFrame(valid)
    print(f"Raw fights  ({start_date[:4]}-{end_date[:4]}): {len(fights)}")
    print(f"After {MIN_PREV_FIGHTS}+ fight filter: {len(filtered)}")
    print(f"Dropped: {len(fights)-len(filtered)} ({(len(fights)-len(filtered))/len(fights)*100:.1f}%)")
    return filtered

base_fights = generate_base_fights_filtered()
print(f"\n✔ Base fights shape: {base_fights.shape}")
print(f"✔ Date range: {base_fights['event_date'].min()} to {base_fights['event_date'].max()}")
print(f"✔ Winner distribution: {base_fights['winner'].value_counts().to_dict()}")


Raw fights  (2014-2026): 5721
After 3+ fight filter: 2837
Dropped: 2884 (50.4%)

✔ Base fights shape: (2837, 12)
✔ Date range: 2014-01-15 to 2026-02-07
✔ Winner distribution: {0: 1450, 1: 1387}


In [16]:
# ============================================================
# CELL 6: Three-Layer Feature Function
# ============================================================

STATS  = ['slpm', 'str_acc', 'td_acc', 'td_avg', 'sub_avg',
          'ctrl_time_per_min', 'kd_per_min']
K_MEAN = 4.0
K_MAD  = 4.0
WINDOW = 5

def calculate_three_layer_features_v2(fighter_id, opponent_id, as_of_date,
                                       stats=STATS, window=WINDOW):
    if fighter_id not in fighter_fights_dict or opponent_id not in fighter_fights_dict:
        return None

    fighter_hist  = fighter_fights_dict[fighter_id]
    opponent_hist = fighter_fights_dict[opponent_id]
    fighter_prev  = fighter_hist[fighter_hist['event_date']   < as_of_date]
    opponent_prev = opponent_hist[opponent_hist['event_date'] < as_of_date]

    if len(fighter_prev) == 0 or len(opponent_prev) == 0:
        return None

    pop_means = {s: float(np.median(all_fight_stats[s].values)) for s in stats}
    pop_mads  = {s: float(np.median(np.abs(
        all_fight_stats[s].values - pop_means[s]
    ))) for s in stats}

    fighter_recent  = fighter_prev.tail(window)
    fighter_weights = time_decay_weights(fighter_recent['event_date'].tolist(), as_of_date)
    fighter_n_eff   = kish_effective_n(fighter_weights)

    features = {}

    for stat in stats:
        decayed_avg   = np.average(fighter_recent[stat].values, weights=fighter_weights)
        smoothed_stat = bayesian_smooth(decayed_avg, fighter_n_eff, pop_means[stat], K_MEAN)
        features[f'{stat}_dec_avg'] = smoothed_stat

        opp_allowed, opp_dates = [], []
        for _, opp_fight in opponent_prev.iterrows():
            opp_opp_id = opponents_dict.get((opp_fight['fight_id'], opponent_id))
            if opp_opp_id and opp_opp_id in fighter_fights_dict:
                opp_opp_fights     = fighter_fights_dict[opp_opp_id]
                opp_opp_this_fight = opp_opp_fights[
                    opp_opp_fights['fight_id'] == opp_fight['fight_id']
                ]
                if len(opp_opp_this_fight) > 0:
                    opp_allowed.append(opp_opp_this_fight[stat].iloc[0])
                    opp_dates.append(opp_fight['event_date'])

        if len(opp_allowed) >= 2:
            opp_weights = time_decay_weights(opp_dates, as_of_date)
            opp_n_eff   = kish_effective_n(opp_weights)
            opp_mean    = np.average(opp_allowed, weights=opp_weights)
            opp_mad     = float(np.median(np.abs(np.array(opp_allowed) - np.median(opp_allowed))))
            opp_mu      = bayesian_smooth(opp_mean, opp_n_eff, pop_means[stat], K_MEAN)
            opp_sigma   = max(bayesian_smooth(opp_mad, opp_n_eff, pop_mads[stat], K_MAD), 0.01)
            features[f'{stat}_opp_dec_avg'] = opp_mu
            features[f'{stat}_opp_mad']     = opp_sigma
            features[f'{stat}_adjperf']     = np.clip((smoothed_stat - opp_mu) / opp_sigma, -7, 7)
        else:
            features[f'{stat}_opp_dec_avg'] = pop_means[stat]
            features[f'{stat}_opp_mad']     = 1.0
            features[f'{stat}_adjperf']     = 0.0

        if fighter_id in fighter_adjperf_history:
            ap_hist  = fighter_adjperf_history[fighter_id]
            ap_prev  = ap_hist[ap_hist['event_date'] < as_of_date].tail(window)
            snap_col = f'{stat}_adjperf_snapshot'
            if len(ap_prev) > 0 and snap_col in ap_prev.columns:
                ap_weights = time_decay_weights(ap_prev['event_date'].tolist(), as_of_date)
                ap_n_eff   = kish_effective_n(ap_weights)
                ap_dec_avg = np.average(ap_prev[snap_col].values, weights=ap_weights)
                features[f'{stat}_dec_adjperf'] = bayesian_smooth(ap_dec_avg, ap_n_eff, 0.0, K_MEAN)
            else:
                features[f'{stat}_dec_adjperf'] = 0.0
        else:
            features[f'{stat}_dec_adjperf'] = 0.0

    return features

print("✓ Three-layer feature function ready")


✓ Three-layer feature function ready


In [17]:


# ============================================================
# CELL 6b: Strike Breakdown Feature Function
# ============================================================

STRIKE_STATS_OFF      = ['head_lpm', 'body_lpm', 'leg_lpm',
                          'distance_lpm', 'clinch_lpm', 'ground_lpm',
                          'head_acc', 'body_acc', 'distance_acc']
STRIKE_STATS_DEF      = ['head_allowed', 'body_allowed', 'leg_allowed',
                          'distance_allowed', 'clinch_allowed', 'ground_allowed']
STRIKE_ADJPERF_CACHED = ['head_lpm', 'body_acc', 'distance_acc',
                          'distance_lpm', 'ground_allowed',
                          'head_acc', 'distance_allowed']

def calculate_strike_features(fighter_id, opponent_id, as_of_date, window=WINDOW):
    features = {}

    off_priors = {}
    for s in STRIKE_STATS_OFF:
        if s in strike_breakdown.columns:
            median = float(strike_breakdown[s].median())
            mad    = float(max(np.median(np.abs(strike_breakdown[s].values - median)), 0.01))
            off_priors[s] = {'mean': median, 'mad': mad}
        else:
            off_priors[s] = {'mean': 0.0, 'mad': 1.0}

    def_priors = {
        s: {
            'mean': float(strike_defense_df[s].median()),
            'mad':  float(max(np.median(np.abs(
                strike_defense_df[s].values - strike_defense_df[s].median()
            )), 0.01))
        } for s in STRIKE_STATS_DEF
    }

    fighter_off_smoothed = {}
    if fighter_id in strike_breakdown_dict:
        hist = strike_breakdown_dict[fighter_id]
        prev = hist[hist['event_date'] < as_of_date].tail(window)
        if len(prev) > 0:
            weights = time_decay_weights(prev['event_date'].tolist(), as_of_date)
            n_eff   = kish_effective_n(weights)
            for s in STRIKE_STATS_OFF:
                if s not in prev.columns:
                    features[f'{s}_dec_avg'] = off_priors[s]['mean']
                    fighter_off_smoothed[s]  = off_priors[s]['mean']
                    continue
                dec_avg  = np.average(prev[s].values, weights=weights)
                smoothed = bayesian_smooth(dec_avg, n_eff, off_priors[s]['mean'], K_MEAN)
                features[f'{s}_dec_avg'] = smoothed
                fighter_off_smoothed[s]  = smoothed
        else:
            for s in STRIKE_STATS_OFF:
                features[f'{s}_dec_avg'] = off_priors[s]['mean']
                fighter_off_smoothed[s]  = off_priors[s]['mean']
    else:
        for s in STRIKE_STATS_OFF:
            features[f'{s}_dec_avg'] = off_priors[s]['mean']
            fighter_off_smoothed[s]  = off_priors[s]['mean']

    fighter_def_smoothed = {}
    if fighter_id in strike_defense_dict:
        hist = strike_defense_dict[fighter_id]
        prev = hist[hist['event_date'] < as_of_date].tail(window)
        if len(prev) > 0:
            weights = time_decay_weights(prev['event_date'].tolist(), as_of_date)
            n_eff   = kish_effective_n(weights)
            for s in STRIKE_STATS_DEF:
                dec_avg  = np.average(prev[s].values, weights=weights)
                smoothed = bayesian_smooth(dec_avg, n_eff, def_priors[s]['mean'], K_MEAN)
                features[f'{s}_dec_avg'] = smoothed
                fighter_def_smoothed[s]  = smoothed
        else:
            for s in STRIKE_STATS_DEF:
                features[f'{s}_dec_avg'] = 0.0
                fighter_def_smoothed[s]  = 0.0
    else:
        for s in STRIKE_STATS_DEF:
            features[f'{s}_dec_avg'] = 0.0
            fighter_def_smoothed[s]  = 0.0

    if opponent_id in strike_breakdown_dict:
        opp_hist = strike_breakdown_dict[opponent_id]
        opp_prev = opp_hist[opp_hist['event_date'] < as_of_date]
        for s in STRIKE_STATS_OFF:
            observed = fighter_off_smoothed[s]
            pop_mean = off_priors[s]['mean']
            pop_mad  = off_priors[s]['mad']
            opp_allowed, opp_dates = [], []
            for _, opp_fight in opp_prev.iterrows():
                opp_opp_id = opponents_dict.get((opp_fight['fight_id'], opponent_id))
                if opp_opp_id and opp_opp_id in strike_breakdown_dict:
                    opp_opp_this = strike_breakdown_dict[opp_opp_id][
                        strike_breakdown_dict[opp_opp_id]['fight_id'] == opp_fight['fight_id']
                    ]
                    if len(opp_opp_this) > 0 and s in opp_opp_this.columns:
                        opp_allowed.append(opp_opp_this[s].iloc[0])
                        opp_dates.append(opp_fight['event_date'])
            if len(opp_allowed) >= 2:
                opp_w     = time_decay_weights(opp_dates, as_of_date)
                opp_n_eff = kish_effective_n(opp_w)
                opp_mean  = np.average(opp_allowed, weights=opp_w)
                opp_mad   = float(np.median(np.abs(np.array(opp_allowed) - np.median(opp_allowed))))
                mu        = bayesian_smooth(opp_mean, opp_n_eff, pop_mean, K_MEAN)
                sigma     = max(bayesian_smooth(opp_mad, opp_n_eff, pop_mad, K_MAD), 0.01)
                features[f'{s}_adjperf'] = np.clip((observed - mu) / sigma, -7, 7)
            else:
                features[f'{s}_adjperf'] = 0.0
    else:
        for s in STRIKE_STATS_OFF:
            features[f'{s}_adjperf'] = 0.0

    if opponent_id in strike_breakdown_dict:
        opp_hist = strike_breakdown_dict[opponent_id]
        opp_prev = opp_hist[opp_hist['event_date'] < as_of_date]
        for s in STRIKE_STATS_DEF:
            observed = fighter_def_smoothed[s]
            pop_mean = def_priors[s]['mean']
            pop_mad  = def_priors[s]['mad']
            opp_allowed, opp_dates = [], []
            for _, opp_fight in opp_prev.iterrows():
                opp_opp_id = opponents_dict.get((opp_fight['fight_id'], opponent_id))
                if opp_opp_id and opp_opp_id in strike_defense_dict:
                    opp_opp_this = strike_defense_dict[opp_opp_id][
                        strike_defense_dict[opp_opp_id]['fight_id'] == opp_fight['fight_id']
                    ]
                    if len(opp_opp_this) > 0:
                        opp_allowed.append(opp_opp_this[s].iloc[0])
                        opp_dates.append(opp_fight['event_date'])
            if len(opp_allowed) >= 2:
                opp_w     = time_decay_weights(opp_dates, as_of_date)
                opp_n_eff = kish_effective_n(opp_w)
                opp_mean  = np.average(opp_allowed, weights=opp_w)
                opp_mad   = float(np.median(np.abs(np.array(opp_allowed) - np.median(opp_allowed))))
                mu        = bayesian_smooth(opp_mean, opp_n_eff, pop_mean, K_MEAN)
                sigma     = max(bayesian_smooth(opp_mad, opp_n_eff, pop_mad, K_MAD), 0.01)
                features[f'{s}_adjperf'] = np.clip((observed - mu) / sigma, -7, 7)
            else:
                features[f'{s}_adjperf'] = 0.0
    else:
        for s in STRIKE_STATS_DEF:
            features[f'{s}_adjperf'] = 0.0

    for s in STRIKE_ADJPERF_CACHED:
        snap_col = f'{s}_adjperf_snapshot'
        if fighter_id in fighter_adjperf_history:
            ap_hist = fighter_adjperf_history[fighter_id]
            ap_prev = ap_hist[ap_hist['event_date'] < as_of_date].tail(window)
            if len(ap_prev) > 0 and snap_col in ap_prev.columns:
                ap_weights = time_decay_weights(ap_prev['event_date'].tolist(), as_of_date)
                ap_n_eff   = kish_effective_n(ap_weights)
                ap_dec_avg = np.average(ap_prev[snap_col].values, weights=ap_weights)
                features[f'{s}_dec_adjperf'] = bayesian_smooth(ap_dec_avg, ap_n_eff, 0.0, K_MEAN)
            else:
                features[f'{s}_dec_adjperf'] = 0.0
        else:
            features[f'{s}_dec_adjperf'] = 0.0

    return features

print("✓ Strike feature function ready")


✓ Strike feature function ready


In [18]:


# ============================================================
# CELL 6c: Rolling Career Stats
# ============================================================

career_wc_priors     = {}
career_global_priors = {}

all_results_list = []
for (fid, fight_id), result in fighter_results_dict.items():
    method = None
    fight_rows = all_fight_stats[all_fight_stats['fight_id'] == fight_id]
    if len(fight_rows) > 0:
        method = fight_rows.iloc[0].get('method', None)
    all_results_list.append({
        'fighter_id':   fid,
        'fight_id':     fight_id,
        'is_win':       1 if result in ('win', 'ko_win') else 0,
        'is_ko':        1 if result == 'ko_win' else 0,
        'is_decision':  1 if method and 'DEC' in str(method).upper() else 0,
    })

results_df = pd.DataFrame(all_results_list)
results_with_wc = results_df.merge(
    all_fight_stats[['fight_id', 'fighter_id', 'weight_class']].drop_duplicates(),
    on=['fight_id', 'fighter_id'], how='left'
)
results_with_wc['wc_norm'] = results_with_wc['weight_class'].apply(normalize_weight_class)

career_global_priors['win'] = results_with_wc['is_win'].mean()
career_global_priors['ko']  = results_with_wc['is_ko'].mean()
career_global_priors['dec'] = results_with_wc['is_decision'].mean()

for wc, grp in results_with_wc[results_with_wc['wc_norm'].notna()].groupby('wc_norm'):
    career_wc_priors[wc] = {
        'win': grp['is_win'].mean(),
        'ko':  grp['is_ko'].mean(),
        'dec': grp['is_decision'].mean(),
    }

fight_decision_dict = {
    (row['fighter_id'], row['fight_id']): row['is_decision']
    for _, row in results_with_wc.iterrows()
}

TAU_WIN      = 25.0
TAU_KO       = 23.0
TAU_DEC      = 20.0
TAU_SUB_LAND = 9.0

def get_career_wc_prior(wc, stat):
    wc_n = normalize_weight_class(wc) if wc else None
    if wc_n and wc_n in career_wc_priors and stat in career_wc_priors[wc_n]:
        return career_wc_priors[wc_n][stat]
    return career_global_priors.get(stat, 0.5)

def calculate_career_stats(fighter_id, opponent_id, fight_id, as_of_date, window=10):
    defaults = {
        'days_since_last_fight': 180,
        'win_ratio':             0.5,
        'win_adjperf':           0.0,
        'ko_rate':               0.0,
        'ko_opp_dec_avg':        0.0,
        'decision_rate':         0.5,
        'sub_landing_rate':      0.0,
        'td_defense':            0.5,
        'td_land_ratio_opp':     0.0,
        'ctrl_ratio_opp':        0.0,
        'sub_att_allowed_pm':    0.0,
        'kd_allowed_pm':         0.0,
    }

    if fighter_id not in fighter_fights_dict:
        return defaults

    hist = fighter_fights_dict[fighter_id]
    prev = hist[hist['event_date'] < as_of_date]
    if len(prev) == 0:
        return defaults

    features = {}
    last_date = prev.iloc[-1]['event_date']
    features['days_since_last_fight'] = (
        datetime.strptime(as_of_date, "%Y-%m-%d") -
        datetime.strptime(last_date,  "%Y-%m-%d")
    ).days

    fight_wc = None
    wc_row = all_fight_stats[all_fight_stats['fight_id'] == fight_id]
    if len(wc_row) > 0:
        fight_wc = wc_row.iloc[0]['weight_class']

    prior_fids = prev['fight_id'].tolist()
    results    = [r for r in [fighter_results_dict.get((fighter_id, fid)) for fid in prior_fids] if r]

    n_fights = len(results)
    if n_fights > 0:
        n_wins = sum(1 for r in results if r in ('win', 'ko_win'))
        n_kos  = sum(1 for r in results if r == 'ko_win')
        n_decs = sum(fight_decision_dict.get((fighter_id, fid), 0) for fid in prior_fids)
        win_prior = get_career_wc_prior(fight_wc, 'win')
        ko_prior  = get_career_wc_prior(fight_wc, 'ko')
        dec_prior = get_career_wc_prior(fight_wc, 'dec')
        features['win_ratio']     = (win_prior * TAU_WIN + n_wins) / (TAU_WIN + n_fights)
        features['ko_rate']       = (ko_prior  * TAU_KO  + n_kos)  / (TAU_KO  + n_fights)
        features['decision_rate'] = (dec_prior * TAU_DEC  + n_decs) / (TAU_DEC  + n_fights)
    else:
        features['win_ratio']     = get_career_wc_prior(fight_wc, 'win')
        features['ko_rate']       = get_career_wc_prior(fight_wc, 'ko')
        features['decision_rate'] = get_career_wc_prior(fight_wc, 'dec')

    win_ratio = features['win_ratio']

    total_sub_landed, total_sub_attempted = 0, 0
    for _, fight_row in prev.iterrows():
        result  = fighter_results_dict.get((fighter_id, fight_row['fight_id']))
        sub_att = fight_row.get('sub_attempts', 0)
        total_sub_attempted += sub_att
        if result == 'win':
            fr = all_fight_stats[all_fight_stats['fight_id'] == fight_row['fight_id']]
            if len(fr) > 0 and 'SUB' in str(fr.iloc[0].get('method', '')).upper():
                total_sub_landed += 1
    sub_land_prior = 0.05
    features['sub_landing_rate'] = (
        (sub_land_prior * TAU_SUB_LAND + total_sub_landed) /
        (TAU_SUB_LAND + total_sub_attempted)
    ) if total_sub_attempted > 0 else sub_land_prior

    if opponent_id in fighter_fights_dict:
        opp_hist = fighter_fights_dict[opponent_id]
        opp_prev = opp_hist[opp_hist['event_date'] < as_of_date]
        opp_win_rates, opp_dates = [], []
        for _, opp_fight in opp_prev.iterrows():
            opp_opp_id = opponents_dict.get((opp_fight['fight_id'], opponent_id))
            if opp_opp_id and opp_opp_id in fighter_fights_dict:
                opp_opp_prev = fighter_fights_dict[opp_opp_id][
                    fighter_fights_dict[opp_opp_id]['event_date'] < opp_fight['event_date']
                ]
                opp_opp_results = [r for r in [
                    fighter_results_dict.get((opp_opp_id, fid))
                    for fid in opp_opp_prev['fight_id'].tolist()
                ] if r]
                if len(opp_opp_results) > 0:
                    opp_n    = len(opp_opp_results)
                    opp_wins = sum(1 for r in opp_opp_results if r in ('win', 'ko_win'))
                    opp_wr   = (get_career_wc_prior(fight_wc, 'win') * TAU_WIN + opp_wins) / (TAU_WIN + opp_n)
                    opp_win_rates.append(opp_wr)
                    opp_dates.append(opp_fight['event_date'])
        if len(opp_win_rates) >= 2:
            opp_w     = time_decay_weights(opp_dates, as_of_date)
            opp_n_eff = kish_effective_n(opp_w)
            opp_mean  = np.average(opp_win_rates, weights=opp_w)
            opp_mad   = float(np.median(np.abs(np.array(opp_win_rates) - np.median(opp_win_rates))))
            mu        = bayesian_smooth(opp_mean, opp_n_eff, 0.5, K_MEAN)
            sigma     = max(bayesian_smooth(opp_mad, opp_n_eff, 0.15, K_MAD), 0.01)
            features['win_adjperf'] = np.clip((win_ratio - mu) / sigma, -7, 7)
        else:
            features['win_adjperf'] = 0.0
    else:
        features['win_adjperf'] = 0.0

    recent = prev.tail(window)
    total_td_att, total_td_land   = 0, 0
    total_sub_att, total_minutes  = 0, 0
    total_kd, total_minutes_kd    = 0, 0
    total_ctrl, total_fight_mins  = 0, 0
    opp_ko_rates, opp_ko_dates    = [], []
    opp_td_ratios, opp_td_dates   = [], []

    for _, fight_row in recent.iterrows():
        opp_id = opponents_dict.get((fight_row['fight_id'], fighter_id))
        if opp_id and opp_id in fighter_fights_dict:
            opp_hist = fighter_fights_dict[opp_id]
            opp_this = opp_hist[opp_hist['fight_id'] == fight_row['fight_id']]
            if len(opp_this) > 0:
                opp_row = opp_this.iloc[0]
                total_td_att     += opp_row['td_attempted']
                total_td_land    += opp_row['td_landed']
                total_sub_att    += opp_row['sub_attempts']
                total_minutes    += opp_row['actual_minutes']
                total_kd         += opp_row['kd_per_min'] * opp_row['actual_minutes']
                total_minutes_kd += opp_row['actual_minutes']
                total_ctrl       += opp_row['ctrl_time_per_min'] * opp_row['actual_minutes']
                total_fight_mins += opp_row['actual_minutes']

                opp_prev_fights = opp_hist[opp_hist['event_date'] < fight_row['event_date']]
                opp_results = [r for r in [
                    fighter_results_dict.get((opp_id, fid))
                    for fid in opp_prev_fights['fight_id'].tolist()
                ] if r]
                if len(opp_results) > 0:
                    opp_n_ko = len(opp_results)
                    opp_kos  = sum(1 for r in opp_results if r == 'ko_win')
                    opp_ko_smoothed = (get_career_wc_prior(fight_wc, 'ko') * TAU_KO + opp_kos) / (TAU_KO + opp_n_ko)
                    opp_ko_rates.append(opp_ko_smoothed)
                    opp_ko_dates.append(fight_row['event_date'])

                if opp_row['td_attempted'] > 0:
                    opp_td_ratios.append(opp_row['td_landed'] / opp_row['td_attempted'])
                    opp_td_dates.append(fight_row['event_date'])

    features['td_defense']        = 1 - (total_td_land / total_td_att) if total_td_att > 0 else 0.5
    features['sub_att_allowed_pm']= total_sub_att / total_minutes if total_minutes > 0 else 0.0
    features['kd_allowed_pm']     = total_kd / total_minutes_kd if total_minutes_kd > 0 else 0.0
    features['ctrl_ratio_opp']    = total_ctrl / total_fight_mins if total_fight_mins > 0 else 0.0

    if len(opp_ko_rates) >= 2:
        opp_w = time_decay_weights(opp_ko_dates, as_of_date)
        features['ko_opp_dec_avg'] = float(np.average(opp_ko_rates, weights=opp_w))
    else:
        features['ko_opp_dec_avg'] = 0.0

    if len(opp_td_ratios) >= 2:
        opp_w = time_decay_weights(opp_td_dates, as_of_date)
        features['td_land_ratio_opp'] = float(np.average(opp_td_ratios, weights=opp_w))
    else:
        features['td_land_ratio_opp'] = 0.0

    return features

print("✔ Career stats function ready")

✔ Career stats function ready


In [19]:

# ============================================================
# CELL 6d: Round 1 Feature Function
# ============================================================

R1_STATS = ['r1_slpm', 'r1_ctrl_per_min', 'r1_td_acc',
            'r1_kd_per_min', 'r1_rev_per_min', 'r1_td_att_per_min',
            'r1_head_lpm', 'r1_body_lpm', 'r1_leg_lpm', 'r1_clinch_lpm']

def calculate_r1_features(fighter_id, opponent_id, as_of_date, window=WINDOW):
    features = {}

    r1_priors = {}
    for s in R1_STATS:
        if s in r1_stats.columns:
            median = float(r1_stats[s].median())
            mad    = float(max(np.median(np.abs(r1_stats[s].values - median)), 0.01))
            r1_priors[s] = {'mean': median, 'mad': mad}
        else:
            r1_priors[s] = {'mean': 0.0, 'mad': 1.0}

    fighter_smoothed = {}
    if fighter_id in r1_stats_dict:
        hist = r1_stats_dict[fighter_id]
        prev = hist[hist['event_date'] < as_of_date].tail(window)
        if len(prev) > 0:
            weights = time_decay_weights(prev['event_date'].tolist(), as_of_date)
            n_eff   = kish_effective_n(weights)
            for s in R1_STATS:
                if s not in prev.columns:
                    features[f'{s}_dec_avg'] = r1_priors[s]['mean']
                    fighter_smoothed[s]      = r1_priors[s]['mean']
                    continue
                dec_avg  = np.average(prev[s].values, weights=weights)
                smoothed = bayesian_smooth(dec_avg, n_eff, r1_priors[s]['mean'], K_MEAN)
                features[f'{s}_dec_avg'] = smoothed
                fighter_smoothed[s]      = smoothed
        else:
            for s in R1_STATS:
                features[f'{s}_dec_avg'] = r1_priors[s]['mean']
                fighter_smoothed[s]      = r1_priors[s]['mean']
    else:
        for s in R1_STATS:
            features[f'{s}_dec_avg'] = r1_priors[s]['mean']
            fighter_smoothed[s]      = r1_priors[s]['mean']

    if opponent_id in r1_stats_dict:
        opp_hist = r1_stats_dict[opponent_id]
        opp_prev = opp_hist[opp_hist['event_date'] < as_of_date]
        for s in R1_STATS:
            observed = fighter_smoothed[s]
            pop_mean = r1_priors[s]['mean']
            pop_mad  = r1_priors[s]['mad']
            opp_allowed, opp_dates = [], []
            for _, opp_fight in opp_prev.iterrows():
                opp_opp_id = opponents_dict.get((opp_fight['fight_id'], opponent_id))
                if opp_opp_id and opp_opp_id in r1_stats_dict:
                    opp_opp_this = r1_stats_dict[opp_opp_id][
                        r1_stats_dict[opp_opp_id]['fight_id'] == opp_fight['fight_id']
                    ]
                    if len(opp_opp_this) > 0 and s in opp_opp_this.columns:
                        opp_allowed.append(opp_opp_this[s].iloc[0])
                        opp_dates.append(opp_fight['event_date'])
            if len(opp_allowed) >= 2:
                opp_w     = time_decay_weights(opp_dates, as_of_date)
                opp_n_eff = kish_effective_n(opp_w)
                opp_mean  = np.average(opp_allowed, weights=opp_w)
                opp_mad   = float(np.median(np.abs(np.array(opp_allowed) - np.median(opp_allowed))))
                mu        = bayesian_smooth(opp_mean, opp_n_eff, pop_mean, K_MEAN)
                sigma     = max(bayesian_smooth(opp_mad, opp_n_eff, pop_mad, K_MAD), 0.01)
                features[f'{s}_adjperf']     = np.clip((observed - mu) / sigma, -7, 7)
                features[f'{s}_opp_dec_avg'] = mu
            else:
                features[f'{s}_adjperf']     = 0.0
                features[f'{s}_opp_dec_avg'] = pop_mean
    else:
        for s in R1_STATS:
            features[f'{s}_adjperf']     = 0.0
            features[f'{s}_opp_dec_avg'] = r1_priors[s]['mean']

    leg_opp_allowed, leg_opp_dates = [], []
    if opponent_id in r1_stats_dict:
        opp_prev = r1_stats_dict[opponent_id]
        opp_prev = opp_prev[opp_prev['event_date'] < as_of_date]
        for _, opp_fight in opp_prev.iterrows():
            opp_opp_id = opponents_dict.get((opp_fight['fight_id'], opponent_id))
            if opp_opp_id and opp_opp_id in strike_breakdown_dict:
                opp_opp_strikes = strike_breakdown_dict[opp_opp_id]
                opp_opp_this    = opp_opp_strikes[
                    opp_opp_strikes['fight_id'] == opp_fight['fight_id']
                ]
                if len(opp_opp_this) > 0:
                    leg_opp_allowed.append(opp_opp_this['leg_lpm'].iloc[0])
                    leg_opp_dates.append(opp_fight['event_date'])
    features['leg_land_r1_opp_dec_avg'] = float(np.average(
        leg_opp_allowed,
        weights=time_decay_weights(leg_opp_dates, as_of_date)
    )) if len(leg_opp_allowed) >= 2 else 0.0

    for s in ['r1_slpm', 'r1_rev_per_min']:
        snap_col = f'{s}_adjperf_snapshot'
        if fighter_id in fighter_adjperf_history:
            ap_hist = fighter_adjperf_history[fighter_id]
            ap_prev = ap_hist[ap_hist['event_date'] < as_of_date].tail(window)
            if len(ap_prev) > 0 and snap_col in ap_prev.columns:
                ap_weights = time_decay_weights(ap_prev['event_date'].tolist(), as_of_date)
                ap_n_eff   = kish_effective_n(ap_weights)
                ap_dec_avg = np.average(ap_prev[snap_col].values, weights=ap_weights)
                features[f'{s}_dec_adjperf'] = bayesian_smooth(ap_dec_avg, ap_n_eff, 0.0, K_MEAN)
            else:
                features[f'{s}_dec_adjperf'] = 0.0
        else:
            features[f'{s}_dec_adjperf'] = 0.0

    return features

print("✔ R1 feature function ready")



✔ R1 feature function ready


In [20]:

# ============================================================
# CELL 7: Generate Features for All Fights
# ============================================================

def generate_features(base_df):
    rows  = []
    total = len(base_df)

    for idx, fight in base_df.iterrows():
        if idx % 500 == 0:
            print(f"  Processing {idx}/{total}...")

        f1_id = fight['fighter_1_id']
        f2_id = fight['fighter_2_id']
        fdate = fight['event_date']
        fid   = fight['fight_id']

        f1_feats = calculate_three_layer_features_v2(f1_id, f2_id, fdate)
        f2_feats = calculate_three_layer_features_v2(f2_id, f1_id, fdate)
        if f1_feats is None or f2_feats is None:
            continue

        f1_feats.update(calculate_strike_features(f1_id, f2_id, fdate))
        f2_feats.update(calculate_strike_features(f2_id, f1_id, fdate))
        f1_feats.update(calculate_r1_features(f1_id, f2_id, fdate))
        f2_feats.update(calculate_r1_features(f2_id, f1_id, fdate))
        f1_feats.update(calculate_career_stats(f1_id, f2_id, fid, fdate))
        f2_feats.update(calculate_career_stats(f2_id, f1_id, fid, fdate))

        row = {'fight_id': fid}
        for k, v in f1_feats.items(): row[f'f1_{k}'] = v
        for k, v in f2_feats.items(): row[f'f2_{k}'] = v
        for k   in f1_feats.keys():   row[f'diff_{k}'] = row[f'f1_{k}'] - row[f'f2_{k}']
        rows.append(row)

    feats_df = pd.DataFrame(rows)
    return base_df.merge(feats_df, on='fight_id', how='inner')

print("Generating features...")
start = time.time()
fight_features = generate_features(base_fights)
print(f"Done in {time.time()-start:.1f}s")
print(f"✔ Shape: {fight_features.shape}")
print(f"✔ Features per fighter: {len([c for c in fight_features.columns if c.startswith('f1_')])}")



Generating features...
  Processing 500/2837...
  Processing 1000/2837...
  Processing 1500/2837...
  Processing 2000/2837...
  Processing 3000/2837...
  Processing 4000/2837...
  Processing 4500/2837...
Done in 968.4s
✔ Shape: (2837, 363)
✔ Features per fighter: 117


In [21]:
# ============================================================
# CELL 8: Physical Features & Experience
# ============================================================

def add_physical_and_experience(df, conn):
    all_ids  = set(df['fighter_1_id'].unique()) | set(df['fighter_2_id'].unique())
    fighters = pd.read_sql(f"""
        SELECT fighter_id, reach, height, dob
        FROM fighters_v2
        WHERE fighter_id IN ({','.join(f"'{fid}'" for fid in all_ids)})
    """, conn)

    fighters['reach_in']  = fighters['reach'].apply(parse_reach)
    fighters['height_in'] = fighters['height'].apply(parse_height)

    for prefix, fid_col in [('f1', 'fighter_1_id'), ('f2', 'fighter_2_id')]:
        df = df.merge(
            fighters[['fighter_id', 'reach_in', 'height_in', 'dob']],
            left_on=fid_col, right_on='fighter_id', how='left'
        ).drop('fighter_id', axis=1).rename(columns={
            'reach_in':  f'{prefix}_reach',
            'height_in': f'{prefix}_height',
            'dob':       f'{prefix}_dob'
        })

    df['f1_age'] = df.apply(lambda r: parse_age(r['f1_dob'], r['event_date']), axis=1)
    df['f2_age'] = df.apply(lambda r: parse_age(r['f2_dob'], r['event_date']), axis=1)

    df['age_diff']    = df['f1_age']    - df['f2_age']
    df['reach_diff']  = df['f1_reach']  - df['f2_reach']
    df['height_diff'] = df['f1_height'] - df['f2_height']
    df['age_ratio']   = df['f1_age']    / df['f2_age']
    df['reach_ratio'] = df['f1_reach']  / df['f2_reach']
    df['height_ratio']= df['f1_height'] / df['f2_height']

    for prefix, fid_col in [('f1', 'fighter_1_id'), ('f2', 'fighter_2_id')]:
        ufc_ages = []
        for _, fight in df.iterrows():
            fid = fight[fid_col]
            if fid in fighter_fights_dict:
                first = fighter_fights_dict[fid].iloc[0]['event_date']
                days  = (datetime.strptime(fight['event_date'], "%Y-%m-%d") -
                         datetime.strptime(first, "%Y-%m-%d")).days
                ufc_ages.append(days / 365.25)
            else:
                ufc_ages.append(0)
        df[f'{prefix}_ufc_age'] = ufc_ages

    df['diff_ufc_age'] = df['f1_ufc_age'] - df['f2_ufc_age']
    df = df.drop(['f1_dob', 'f2_dob'], axis=1)
    return df

fight_features = add_physical_and_experience(fight_features, conn)
print(f"✔ Shape with physical features: {fight_features.shape}")

✔ Shape with physical features: (2837, 378)


In [22]:

# ============================================================
# CELL 8b: Interaction Features
# ============================================================

fight_features['age_td_interaction']         = fight_features['age_diff'] * fight_features['diff_td_avg_dec_avg']
fight_features['age_td_adjperf_interaction'] = fight_features['age_diff'] * fight_features['diff_td_avg_adjperf']
fight_features['age_str_interaction']        = fight_features['age_diff'] * fight_features['diff_str_acc_dec_avg']
fight_features['age_winratio_interaction']   = fight_features['age_diff'] * fight_features['diff_win_ratio']
fight_features['ufc_age_td_interaction']     = fight_features['diff_ufc_age'] * fight_features['diff_td_avg_dec_avg']
fight_features['ufc_age_str_interaction']    = fight_features['diff_ufc_age'] * fight_features['diff_str_acc_dec_avg']
fight_features['td_ctrl_interaction']        = fight_features['diff_td_avg_dec_avg'] * fight_features['diff_ctrl_time_per_min_dec_avg']
fight_features['str_headdef_interaction']    = fight_features['diff_str_acc_dec_avg'] * fight_features['diff_head_allowed_dec_avg']

interaction_cols = [
    'age_td_interaction', 'age_td_adjperf_interaction',
    'age_str_interaction', 'age_winratio_interaction',
    'ufc_age_td_interaction', 'ufc_age_str_interaction',
    'td_ctrl_interaction', 'str_headdef_interaction'
]
fight_features[interaction_cols] = fight_features[interaction_cols].fillna(0)

print(f"✓ Added {len(interaction_cols)} interaction features")
print(f"✓ fight_features shape: {fight_features.shape}")


✓ Added 8 interaction features
✓ fight_features shape: (2837, 386)


In [23]:

# ============================================================
# CELL 9: Optuna Hyperparameter Search + Feature Trimming
# ============================================================

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# --- Feature cols ---
FEATURE_COLS = [c for c in fight_features.columns
                if c.startswith('diff_')
                or c in ['age_diff', 'age_ratio', 'reach_diff',
                         'height_diff', 'reach_ratio', 'height_ratio']]

FEATURE_COLS = [c for c in FEATURE_COLS
                if 'strike_share'     not in c
                and 'exchange_ratio'  not in c
                and 'targeting_share' not in c]

# --- Three-way split ---
train = fight_features[fight_features['event_date'] <  '2023-01-01']
val   = fight_features[(fight_features['event_date'] >= '2023-01-01') &
                        (fight_features['event_date'] <  '2025-01-01')]
test  = fight_features[fight_features['event_date'] >= '2025-01-01']

X_train, y_train = train[FEATURE_COLS].fillna(0), train['winner']
X_val,   y_val   = val[FEATURE_COLS].fillna(0),   val['winner']
X_test,  y_test  = test[FEATURE_COLS].fillna(0),  test['winner']

print(f"✓ Features : {len(FEATURE_COLS)}")
print(f"✓ Train    : {X_train.shape}  (<2023)")
print(f"✓ Val      : {X_val.shape}  (2023-2024)")
print(f"✓ Test     : {X_test.shape}  (2025+)")

# --- Feature trimming ---
model_for_imp = XGBClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.05,
    min_child_weight=15, subsample=0.6, colsample_bytree=0.5,
    gamma=2.0, reg_alpha=1.0, reg_lambda=2.0, random_state=42
)
model_for_imp.fit(X_train, y_train)

feat_imp = pd.Series(
    model_for_imp.feature_importances_, index=X_train.columns
).sort_values(ascending=False)

THRESHOLD = 0.005
low_imp = feat_imp[feat_imp < THRESHOLD].index.tolist()

X_train_trimmed = X_train.drop(columns=low_imp)
X_val_trimmed   = X_val.drop(columns=low_imp)
X_test_trimmed  = X_test.drop(columns=low_imp)

print(f"✓ Trimmed  : {X_train_trimmed.shape[1]} features (dropped {len(low_imp)})")
print(f"\nTop 20 features by importance:")
print(feat_imp.head(20).to_string())

# --- Optuna search ---
def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 2, 4), 
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 50),
        'subsample':        trial.suggest_float('subsample', 0.4, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 0.8),
        'gamma':            trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0.0, 5.0),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.5, 5.0),
        'random_state':     42,
    }
    m = XGBClassifier(**params)
    m.fit(X_train_trimmed, y_train)
    val_acc   = m.score(X_val_trimmed, y_val)
    train_acc = m.score(X_train_trimmed, y_train)
    gap = train_acc - val_acc
    if gap > 0.12:
        return val_acc - (gap - 0.12) * 0.5
    return val_acc

print(f"\n✓ Starting Optuna search (300 trials)...")
start = time.time()
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=300)
print(f"Done in {time.time()-start:.1f}s")

# --- Best trial ---
best = study.best_trial
print(f"\n{'='*55}")
print(f"BEST TRIAL: #{best.number}")
print(f"{'='*55}")
print(f"Val accuracy: {best.value:.4f} ({best.value*100:.1f}%)")
print(f"\nBest parameters:")
for k, v in best.params.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

# --- Retrain with best params ---
best_model = XGBClassifier(**best.params, random_state=42)
best_model.fit(X_train_trimmed, y_train)

train_acc = best_model.score(X_train_trimmed, y_train)
val_acc   = best_model.score(X_val_trimmed,   y_val)
test_acc  = best_model.score(X_test_trimmed,  y_test)

print(f"\n{'='*55}")
print(f"FINAL RESULTS")
print(f"{'='*55}")
print(f"Train      : {train_acc:.1%}")
print(f"Val        : {val_acc:.1%}")
print(f"Test       : {test_acc:.1%}")
print(f"Train/Val  : {(train_acc - val_acc)*100:+.1f}%")
print(f"Val/Test   : {(val_acc  - test_acc)*100:+.1f}%")

# --- Per-year breakdown ---
print(f"\n{'='*55}")
print(f"PER YEAR BREAKDOWN")
print(f"{'='*55}")
print(f"{'Year':<6} {'Acc':>8} {'N':>6} {'Split':>8}")
print("─" * 32)
trimmed_cols = X_train_trimmed.columns.tolist()
for year in [2021, 2022, 2023, 2024, 2025, 2026]:
    year_data = fight_features[
        (fight_features['event_date'] >= f'{year}-01-01') &
        (fight_features['event_date'] <  f'{year+1}-01-01')
    ]
    if len(year_data) == 0:
        continue
    X_y = year_data[trimmed_cols].fillna(0)
    y_y = year_data['winner']
    acc = best_model.score(X_y, y_y)
    split = 'train' if year < 2023 else ('val' if year < 2025 else 'test')
    print(f"{year:<6} {acc:>8.1%} {len(year_data):>6} {split:>8}")

# --- Top 5 trials ---
print(f"\n{'='*55}")
print(f"TOP 5 TRIALS")
print(f"{'='*55}")
for t in sorted(study.trials, key=lambda t: t.value, reverse=True)[:5]:
    m = XGBClassifier(**t.params, random_state=42)
    m.fit(X_train_trimmed, y_train)
    ta = m.score(X_train_trimmed, y_train)
    print(f"  Trial {t.number:>3}: val={t.value:.4f}  train={ta:.4f}  gap={ta - t.value:.4f}")

model_trimmed = best_model




C:\Users\Sarthak\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Features : 124
✓ Train    : (1957, 124)  (<2023)
✓ Val      : (551, 124)  (2023-2024)
✓ Test     : (329, 124)  (2025+)
✓ Trimmed  : 118 features (dropped 6)

Top 20 features by importance:
age_ratio                             0.016650
age_diff                              0.014967
diff_head_allowed_dec_avg             0.014581
diff_ground_allowed_dec_adjperf       0.014292
diff_r1_slpm_opp_dec_avg              0.013567
diff_ctrl_ratio_opp                   0.013169
diff_ctrl_time_per_min_opp_dec_avg    0.012247
diff_str_acc_opp_dec_avg              0.012214
diff_win_ratio                        0.011998
diff_kd_per_min_opp_dec_avg           0.011642
diff_distance_acc_dec_adjperf         0.011548
diff_win_adjperf                      0.011355
diff_str_acc_dec_adjperf              0.010913
diff_td_land_ratio_opp                0.010679
diff_r1_slpm_dec_adjperf              0.010654
diff_head_acc_dec_avg                 0.010593
diff_r1_ctrl_per_min_opp_dec_avg      0.010419
diff_r1_td

In [24]:
import pickle

pickle.dump(all_fight_stats, open('data/all_fight_stats.pkl', 'wb'))
pickle.dump(strike_breakdown, open('data/strike_breakdown.pkl', 'wb'))
pickle.dump(adjperf_history_df, open('data/adjperf_history.pkl', 'wb'))
pickle.dump(fight_features, open('data/fight_features.pkl', 'wb'))
pickle.dump(fighter_fights_dict, open('data/fighter_fights_dict.pkl', 'wb'))
pickle.dump(opponents_dict, open('data/opponents_dict.pkl', 'wb'))
pickle.dump(fighter_adjperf_history, open('data/fighter_adjperf_history.pkl', 'wb'))
print("✔ All data saved")

✔ All data saved


In [25]:
import pickle

DATA_PATH = r'C:\Users\Sarthak\Documents\ML\fighter-beta\mma\notebooks\02_features\data'

pickle.dump(best_model, open(f'{DATA_PATH}/xgb_best_model.pkl', 'wb'))
pickle.dump(X_val_trimmed.columns.tolist(), open(f'{DATA_PATH}/feature_cols.pkl', 'wb'))
print(f"✔ Model saved. Val: {val_acc:.1%} Test: {test_acc:.1%}")

✔ Model saved. Val: 66.1% Test: 63.2%


In [26]:
pickle.dump(strike_breakdown_dict, open(f'{DATA_PATH}/strike_breakdown_dict.pkl', 'wb'))
pickle.dump(strike_defense_dict, open(f'{DATA_PATH}/strike_defense_dict.pkl', 'wb'))
pickle.dump(r1_stats_dict, open(f'{DATA_PATH}/r1_stats_dict.pkl', 'wb'))
pickle.dump(fighter_results_dict, open(f'{DATA_PATH}/fighter_results_dict.pkl', 'wb'))
pickle.dump(fight_decision_dict, open(f'{DATA_PATH}/fight_decision_dict.pkl', 'wb'))
print("✔ Additional dicts saved")

✔ Additional dicts saved
